<a href="https://colab.research.google.com/github/swapna0727-source/AI/blob/main/Pydantic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import time
import os

import os

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError(
        "OPENROUTER_API_KEY not found. Please add it to your Google Colab Secrets "
        "(the key icon on the left) with the name 'OPENROUTER_API_KEY' and enable notebook access."
    )

print("Key loaded:", OPENROUTER_API_KEY[:8] + "...")  # never print the full key

OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
url = "https://openrouter.ai/api/v1/chat/completions"

def ask_llm(user_message, system_message=None, model="openai/gpt-4o-mini", temperature=0.7):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }

    messages = []
    if system_message:
        messages.append({"role": "system", "content": system_message})
    messages.append({"role": "user", "content": user_message})

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
    }

    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
        data = response.json()
        return data["choices"][0]["message"]["content"]
    elif response.status_code == 429:
        print("Rate limited, retrying once after 2s...")
        time.sleep(2)
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            return response.json()["choices"][0]["message"]["content"]
        print(f"Retry also failed: {response.status_code}")
        return None
    else:
        print(f"Request failed: {response.status_code} - {response.text}")
        return None

# Test it
print(ask_llm("What is 12 * 8?"))
print(ask_llm(
    "Translate 'good morning' to French.",
    system_message="You are a translation assistant. Reply with only the translation, nothing else."
))

Key loaded: sk-or-v1...
12 * 8 = 96.
Bonjour


In [ ]:
prompt = """Extract the customer's name, order number, and issue category
(one of: SHIPPING, BILLING, DEFECTIVE_ITEM, OTHER) from this message,
as JSON with keys name, order_number, issue_category.

Message: "Hi, this is Rohan Verma, order #48213 — the item arrived late."
"""

print(ask_llm(prompt))

```json
{
  "name": "Rohan Verma",
  "order_number": "48213",
  "issue_category": "SHIPPING"
}
```


In [ ]:
from pydantic import BaseModel

In [ ]:
class SupportTicket(BaseModel):
    customer_name: str
    order_number: int
    issue_description: str


In [ ]:
ticket = SupportTicket(
    customer_name="Rohan Verma",
    order_number=48213,
    issue_description="Item arrived cracked",
)

In [ ]:
print(ticket)

customer_name='Rohan Verma' order_number=48213 issue_description='Item arrived cracked'


In [ ]:
print(type(ticket))

<class '__main__.SupportTicket'>


In [ ]:
ticket2 = SupportTicket(
    customer_name="Anita Rao",
    order_number="48910",  # a string here, but the field expects int
    issue_description="Billing question",
)
print(ticket2.order_number, type(ticket2.order_number))

48910 <class 'int'>


In [ ]:
ticket3 = SupportTicket(
    customer_name="Anita Rao",
    order_number="A48910",  # a string here, but the field expects int
    issue_description="Billing question",
)
print(ticket2.order_number, type(ticket2.order_number))

ValidationError: 1 validation error for SupportTicket
order_number
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='A48910', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

In [ ]:
from typing import Literal

In [ ]:
class SupportTicketV2(BaseModel):
    customer_name: str
    order_number: int
    issue_category: Literal["SHIPPING", "BILLING", "DEFECTIVE_ITEM", "OTHER"]


In [ ]:
ticket3 = SupportTicketV2(
    customer_name="Anita Rao",
    order_number=48910,  # a string here, but the field expects int
    issue_category="BILLING",
)

In [ ]:
from pydantic import Field

In [ ]:
class SupportTicketV3(BaseModel):
    customer_name: str
    order_number: int
    issue_category: Literal["SHIPPING", "BILLING", "DEFECTIVE_ITEM", "OTHER"]
    urgency_score: int = Field(ge=1, le=5, description="Urgency from 1 (low) to 5 (critical)")
    requires_manager: bool = False  # default value if not provided


In [ ]:
t = SupportTicketV3(
    customer_name="Priya Nair",
    order_number=91022,
    issue_category="BILLING",
    urgency_score=4,
)
print(t)

customer_name='Priya Nair' order_number=91022 issue_category='BILLING' urgency_score=4 requires_manager=False


In [ ]:
json_string = '{"customer_name": "Anita Rao", "order_number": 55210, "issue_category": "SHIPPING", "urgency_score": 2}'

ticket_from_json = SupportTicketV3.model_validate_json(json_string)
print(ticket_from_json)

data_dict = {"customer_name": "Karan Mehta", "order_number": 33012, "issue_category": "OTHER", "urgency_score": 1}
ticket_from_dict = SupportTicketV3.model_validate(data_dict)
print(ticket_from_dict)

customer_name='Anita Rao' order_number=55210 issue_category='SHIPPING' urgency_score=2 requires_manager=False
customer_name='Karan Mehta' order_number=33012 issue_category='OTHER' urgency_score=1 requires_manager=False


In [ ]:
print(type(ticket_from_dict))

<class '__main__.SupportTicketV3'>


In [ ]:
print(ticket_from_dict.model_dump_json())
print(ticket_from_json.model_dump_json())

{"customer_name":"Karan Mehta","order_number":33012,"issue_category":"OTHER","urgency_score":1,"requires_manager":false}
{"customer_name":"Anita Rao","order_number":55210,"issue_category":"SHIPPING","urgency_score":2,"requires_manager":false}


In [ ]:
from pydantic import ValidationError

bad_data = {
    "customer_name": "Sanya Kapoor",
    "order_number": "not-a-number",
    "issue_category": "DAMAGED",  # not one of our four allowed values
    "urgency_score": 9,           # out of the 1-5 range
}

try:
    ticket = SupportTicketV3.model_validate(bad_data)
except ValidationError as e:
    print(e)

3 validation errors for SupportTicketV3
order_number
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='not-a-number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing
issue_category
  Input should be 'SHIPPING', 'BILLING', 'DEFECTIVE_ITEM' or 'OTHER' [type=literal_error, input_value='DAMAGED', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error
urgency_score
  Input should be less than or equal to 5 [type=less_than_equal, input_value=9, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


In [ ]:
try:
    ticket = SupportTicketV3.model_validate(bad_data)
except ValidationError as e:
    for error in e.errors():
        print(f"Field: {error['loc']}, Problem: {error['msg']}")

Field: ('order_number',), Problem: Input should be a valid integer, unable to parse string as an integer
Field: ('issue_category',), Problem: Input should be 'SHIPPING', 'BILLING', 'DEFECTIVE_ITEM' or 'OTHER'
Field: ('urgency_score',), Problem: Input should be less than or equal to 5


In [ ]:
!pip install instructor

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.2/353.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: jiter
    Found existing installation: jiter 0.17.0
    Uninstalling jiter-0.17.0:
      Successfully uninstalled jiter-0.17.0
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests

In [ ]:
import os
import instructor
from openai import OpenAI
from google.colab import userdata

In [ ]:
base_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY"),
)


In [ ]:
response = base_client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Hi, this is Rohan Verma, order #48213 — the item arrived late."}
    ],

)

In [ ]:
type(response)

openai.types.chat.chat_completion.ChatCompletion

In [ ]:
client = instructor.from_openai(base_client)

In [ ]:
from pydantic import BaseModel
from typing import Literal

class ExtractedTicket(BaseModel):
    customer_name: str
    order_number: int
    issue_category: Literal["SHIPPING", "BILLING", "DEFECTIVE_ITEM", "OTHER"]

result = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    response_model=ExtractedTicket,
    messages=[
        {"role": "user", "content": "Hi, this is Rohan Verma, order #48213 — the item arrived cracked."}
    ],
)

print(result)
print(type(result))
print(result.issue_category)

customer_name='Rohan Verma' order_number=48213 issue_category='DEFECTIVE_ITEM'
<class '__main__.ExtractedTicket'>
DEFECTIVE_ITEM


In [ ]:
from pydantic import ValidationError

def extract_ticket_safely(message_text, max_retries=2):
    try:
        result = client.chat.completions.create(
            model="openai/gpt-4o-mini",
            response_model=ExtractedTicket,
            max_retries=max_retries,  # instructor's own built-in retry-on-validation-failure count
            messages=[{"role": "user", "content": message_text}],
        )
        return result
    except ValidationError as e:
        print(f"Could not extract a valid ticket after retries. Errors: {e.errors()}")
        return None
    except Exception as e:
        print(f"Unexpected error during extraction: {e}")
        return None

ticket = extract_ticket_safely("asdkj not really a support message at all, just noise")
if ticket is None:
    print("Falling back to manual review queue.")
else:
    print(ticket)

customer_name='asdkj' order_number=0 issue_category='OTHER'


In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

class FeedbackEntry(BaseModel):
    customer_sentiment: Literal[___, ___, ___]
    product_mentioned: ___
    follow_up_needed: ___
    priority: int = Field(ge=___, le=___)

def extract_feedback_safely(text, max_retries=2):
    try:
        result = client.chat.completions.create(
            model="openai/gpt-4o-mini",
            response_model=___,
            max_retries=___,
            messages=[{"role": "user", "content": ___}],
        )
        return result
    except ValidationError as e:
        print(f"Validation failed: {___}")
        return None

# Test cases
well_formed = "The new blender is great but shipping took way too long -- 10 days! Might return it if it happens again."
vague = "hmm ok"

print(extract_feedback_safely(well_formed))
print(extract_feedback_safely(vague))